In [1]:
!pip install lightning

In [2]:
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
import os

os.chdir("/content/drive/Othercomputers/Laptop HP/ta_module/Coding")

In [4]:
import pandas as pd
from pathlib import Path

In [5]:
cwd = Path.cwd()
print(cwd)

/content/drive/Othercomputers/Laptop HP/TA/Coding


In [6]:
mortalitas_df = pd.read_csv(cwd / "data/mortalitas.csv", parse_dates=["Year"], date_format="%Y")
populasi_df = pd.read_csv(cwd / "data/populasi.csv")
bi_rate_df = pd.read_csv(cwd / "data/bi_rate.csv", parse_dates=["Date"], date_format="%d-%m-%y")

In [7]:
mortalitas_df.head()

,Year,Sex,Age,Value
0,1950-01-01,Male,0,0.228515
1,1950-01-01,Female,0,0.185158
2,1950-01-01,Male,1,0.069349
3,1950-01-01,Female,1,0.063058
4,1950-01-01,Male,2,0.044724


In [8]:
mortalitas_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15150 entries, 0 to 15149
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Year    15150 non-null  datetime64[ns]
 1   Sex     15150 non-null  object        
 2   Age     15150 non-null  int64         
 3   Value   15150 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1), object(1)
memory usage: 473.6+ KB


In [9]:
populasi_df.head()

,Sex,Age,Value
0,Male,45,2013106
1,Male,46,1965078
2,Male,47,1918556
3,Male,48,1880902
4,Male,49,1836713


In [10]:
populasi_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Sex     32 non-null     object
 1   Age     32 non-null     int64 
 2   Value   32 non-null     int64 
dtypes: int64(2), object(1)
memory usage: 900.0+ bytes


In [11]:
bi_rate_df.head()

,Date,Value
0,2024-12-18,0.0600
1,2024-11-20,0.0600
2,2024-10-16,0.0600
3,2024-09-18,0.0600
4,2024-08-21,0.0625


In [12]:
bi_rate_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    60 non-null     datetime64[ns]
 1   Value   60 non-null     float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 1.1 KB


In [13]:
AGE_MIN = min(mortalitas_df["Age"])
AGE_MAX = max(mortalitas_df["Age"])

YEAR_MIN = min(mortalitas_df["Year"])
YEAR_MAX = max(mortalitas_df["Year"])

# Analisis karakteristik data

In [14]:
import seaborn as sns

sns.set_theme(style="whitegrid")

## Plot mortalitas satu usia untuk semua tahun

In [15]:
def plot_usia_vs_tahun(
        age_start=AGE_MIN,
        age_end=AGE_MAX,
        dir=cwd / "plot"
):
    plotname = f"usia vs tahun ({age_start}-{age_end})"
    filepath = dir / f"{plotname}.png"
    if not filepath.exists():
        mask = (mortalitas_df["Age"] >= age_start) & (mortalitas_df["Age"] <= age_end)
        g = sns.FacetGrid(
            mortalitas_df[mask],
            col="Age",
            hue="Sex",
            height=5,
            col_wrap=5,
            sharex=False,

            sharey=False
        )

        g.map_dataframe(sns.lineplot, x="Year", y="Value")

        g.set_titles("Age {col_name}")
        g.set_axis_labels("Year", "Mortality Rate")
        g.add_legend()

        g.tight_layout()
        g.savefig(filepath)
        print(f"Plot {plotname} saved to {filepath}")

In [16]:
plot_usia_vs_tahun(AGE_MIN, 20)
plot_usia_vs_tahun(21, 40)
plot_usia_vs_tahun(41, 60)
plot_usia_vs_tahun(61, 80)
plot_usia_vs_tahun(81, AGE_MAX)

## Plot mortalitas semua usia untuk satu tahun

In [17]:
def plot_tahun_vs_usia(
        year_start=YEAR_MIN,
        year_end=YEAR_MAX,
        dir=cwd / "plot"
):
    plotname = f"tahun vs usia ({year_start}-{year_end})"
    filepath = dir / f"{plotname}.png"
    if not filepath.exists():
        mask = (mortalitas_df["Year"] >= year_start) & (mortalitas_df["Year"] <= year_end)
        df = mortalitas_df.copy()
        df["Year Only"] = df["Year"].dt.year

        g = sns.FacetGrid(
            df[mask],
            col="Year Only",
            hue="Sex",
            height=5,
            col_wrap=5,
            sharex=False,
            sharey=False
        )

        g.map_dataframe(sns.lineplot, x="Age", y="Value")

        g.set_titles("Year {col_name}")
        g.set_axis_labels("Age", "Mortality Rate")
        g.add_legend()

        g.tight_layout()
        g.savefig(filepath)
        print(f"Plot {plotname} saved to {filepath}")

In [18]:
plot_tahun_vs_usia("1950", "1964")
plot_tahun_vs_usia("1965", "1979")
plot_tahun_vs_usia("1980", "1994")
plot_tahun_vs_usia("1995", "2009")
plot_tahun_vs_usia("2010", "2024")

# Pemodelan LocalGLMnet

## Pemodelan

In [19]:
import torch.nn as nn

from ta_module.data import MortalityDataset, MortalityDataModule
from ta_module.models import LocalGLMnet, LocallyConnected2D, MyModel
from ta_module.utils import RegularizationLoss, IdentityTransform
from torch.optim import NAdam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch import seed_everything

import lightning as L

In [20]:
seed_everything(42, workers=True)

INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42


42

In [21]:
lookback = 10
horizon = 1

mortalitas_df_male = mortalitas_df[mortalitas_df["Sex"] == "Male"]
mortalitas_dataset_male = MortalityDataset(
    df=mortalitas_df_male,
    lookback=lookback,
    horizon=horizon,
    age_col="Age",
    mortality_col="Value",
    year_col="Year"
)

mortalitas_df_female = mortalitas_df[mortalitas_df["Sex"] == "Female"]
mortalitas_dataset_female = MortalityDataset(
    df=mortalitas_df_female,
    lookback=lookback,
    horizon=horizon,
    age_col="Age",
    mortality_col="Value",
    year_col="Year"
)

In [22]:
train_val_test_splits = (0.95, 0.05, 0.0)
batch_size = 16

mortalitas_datamodule = MortalityDataModule(
    mortality_datasets=[mortalitas_dataset_male, mortalitas_dataset_female],
    batch_sizes=batch_size,
    splits=train_val_test_splits
)

In [23]:
ukuran_matriks_input = (lookback, AGE_MAX - AGE_MIN + 1)
lcn_activation_function = nn.Sigmoid()
lcn_kernel_size = 5

create_lcn_layer = LocallyConnected2D.factory(
    input_size=ukuran_matriks_input,
    kernel_size=lcn_kernel_size,
    activation_fn=lcn_activation_function,
    zero_padding=True
)

create_localglmnet_model = LocalGLMnet.factory(
    input_size=ukuran_matriks_input,
    link_fn=IdentityTransform()
)

In [24]:
num_ensembles = 10
localglmnet_models = [
    create_localglmnet_model(regression_attention_model=create_lcn_layer())
    for _ in range(num_ensembles)
]

In [25]:


from ta_module.utils import rmse_loss

In [26]:
train_loss_fn = rmse_loss
eval_loss_fn = rmse_loss

regularization_parameter = 10e-5
create_regularization_loss = RegularizationLoss.factory(
    eta=10e-5,
    alfa=1
)

In [27]:
from typing import Iterator
import torch

In [28]:
def _nadam_optimizer_factory(*args, **kwargs):
    def create_nadam_optimizer(params: Iterator[nn.Parameter]):
        return NAdam(
            params,
            *args,
            **kwargs
        )

    return create_nadam_optimizer


def _reduce_lr_on_plateu_scheduler_factory(*args, **kwargs):
    def create_reduce_lr_on_plateu_scheduler(optimizer: torch.optim.Optimizer):
        return ReduceLROnPlateau(
            optimizer,
            *args,
            **kwargs
        )

    return create_reduce_lr_on_plateu_scheduler

In [29]:
optimizer_factory = _nadam_optimizer_factory(lr=1e-3)
lr_scheduler_factory = _reduce_lr_on_plateu_scheduler_factory(
    factor=0.90,
    patience=50,
    min_lr=5e-5,
    cooldown=5,
    mode="min"
)

In [30]:
localglmnet_ensembles = [
    MyModel(
        model=m,
        train_loss=train_loss_fn,
        eval_loss=eval_loss_fn,
        regularization_loss=create_regularization_loss(
            model_weights_getter=(lambda: m.regression_attention_model.parameters())
        ),
        optimizer_factory=optimizer_factory,
        lr_scheduler_factory=lr_scheduler_factory
    )

    for m in localglmnet_models
]

In [31]:
from datetime import datetime
from zoneinfo import ZoneInfo

In [32]:
from lightning.pytorch.callbacks import EarlyStopping

In [33]:
timezone = ZoneInfo("Asia/Jakarta")
run_datetime = datetime.now(tz=timezone).strftime("%Y-%m-%d_%H-%M-%S")

checkpoint_dir = "best_model_checkpoints"
checkpoint_filenames = [f"Model_LocalGLMnet_{i + 1}_{run_datetime}" for i in range(num_ensembles)]

num_epochs = 500
trainers = [
    L.Trainer(
        max_epochs=num_epochs,
        callbacks=[
            ModelCheckpoint(
                dirpath=checkpoint_dir,
                filename=checkpoint_filenames[i],
                monitor="val_loss",
                mode="min",
                save_top_k=1
            ),
            EarlyStopping(
                min_delta=1e-4,
                monitor="val_loss",
                mode="min",
                patience=5,
            )
        ],
        log_every_n_steps=1,
        deterministic=True
    )
    for i in range(num_ensembles)
]

INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU c

In [34]:
# for i, (trainer, model) in enumerate(zip(trainers, localglmnet_ensembles)):
#     print("\n=============================")
#     print(i)
#     print(trainer)
#     print(model)

In [35]:
# Train ensembles pada data mortalitas laki-laki dan perempuan
print(f"Run datetime: {run_datetime}")
for i, (trainer, model) in enumerate(zip(trainers, localglmnet_ensembles)):
    print("\n================================================================")
    print(f"Training model {i + 1} pada mortalitas laki-laki dan perempuan:")
    print("================================================================\n")
    trainer.fit(
        model=model,
        datamodule=mortalitas_datamodule,
    )

Run datetime: 2026-03-31_12-35-40

Training model 1 pada mortalitas laki-laki dan perempuan:



/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /content/drive/Othercomputers/Laptop HP/TA/Coding/best_model_checkpoints exists and is not empty.


┏━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model               │ LocalGLMnet        │ 26.4 K │ train │     0 │
│ 1 │ regularization_loss │ RegularizationLoss │      0 │ train │     0 │
└───┴─────────────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 26.4 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 26.4 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 4                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

...

Output()


Training model 10 pada mortalitas laki-laki dan perempuan:



┏━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model               │ LocalGLMnet        │ 26.4 K │ train │     0 │
│ 1 │ regularization_loss │ RegularizationLoss │      0 │ train │     0 │
└───┴─────────────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 26.4 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 26.4 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 4                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

In [37]:
localglmnet_ensembles_best = [
    MyModel.load_from_checkpoint(
        f"{checkpoint_dir}/{f}.ckpt",
        weights_only=False,
        model=m,
        train_loss=train_loss_fn,
        eval_loss=eval_loss_fn,
        regularizaton_loss=create_regularization_loss(
            model_weights_getter=(lambda: m.regression_attention_model.parameters())
        ),
        optimizer_factory=optimizer_factory,
        lr_scheduler_factory=lr_scheduler_factory
    )
    for f, m in zip(checkpoint_filenames,
                    [create_localglmnet_model(regression_attention_model=create_lcn_layer()) for _ in
                     range(num_ensembles)])
]
print(localglmnet_ensembles_best)

[MyModel(
  (model): LocalGLMnet(
    (regression_attention_model): LocallyConnected2D(
      (activation_fn): Sigmoid()
    )
  )
), MyModel(
  (model): LocalGLMnet(
    (regression_attention_model): LocallyConnected2D(
      (activation_fn): Sigmoid()
    )
  )
), MyModel(
  (model): LocalGLMnet(
    (regression_attention_model): LocallyConnected2D(
      (activation_fn): Sigmoid()
    )
  )
), MyModel(
  (model): LocalGLMnet(
    (regression_attention_model): LocallyConnected2D(
      (activation_fn): Sigmoid()
    )
  )
), MyModel(
  (model): LocalGLMnet(
    (regression_attention_model): LocallyConnected2D(
      (activation_fn): Sigmoid()
    )
  )
), MyModel(
  (model): LocalGLMnet(
    (regression_attention_model): LocallyConnected2D(
      (activation_fn): Sigmoid()
    )
  )
), MyModel(
  (model): LocalGLMnet(
    (regression_attention_model): LocallyConnected2D(
      (activation_fn): Sigmoid()
    )
  )
), MyModel(
  (model): LocalGLMnet(
    (regression_attention_model): L

In [55]:
print("Cek apakah parameter di localglmnet_ensembles_best benar sama dengan parameter di checkpoint:")

for i, (c, m_1, m_2) in enumerate(zip(checkpoints, localglmnet_ensembles, localglmnet_ensembles_best)):
    print(f"\nModel {i + 1}")
    print("=============================================")
    checkpoint_state_dict = c["state_dict"]
    for (n_1, p_1), (n_2, p_2) in zip(m_1.named_parameters(), m_2.named_parameters()):
        print(f"{n_1}\n")
        print("Trained vs Checkpoint:", torch.equal(checkpoint_state_dict[n_1], p_1))
        print("Best vs Checkpoint:", torch.equal(checkpoint_state_dict[n_2], p_2))
        print("=============================================")

Cek apakah parameter di localglmnet_ensembles_best benar sama dengan parameter di checkpoint:

Model 1
model.bias

Trained vs Checkpoint: False
Best vs Checkpoint: True
model.regression_attention_model.weight

Trained vs Checkpoint: False
Best vs Checkpoint: True
model.regression_attention_model.bias

Trained vs Checkpoint: False
Best vs Checkpoint: True

Model 2
model.bias

Trained vs Checkpoint: False
Best vs Checkpoint: True
model.regression_attention_model.weight

Trained vs Checkpoint: False
Best vs Checkpoint: True
model.regression_attention_model.bias

Trained vs Checkpoint: False
Best vs Checkpoint: True

Model 3
model.bias

Trained vs Checkpoint: False
Best vs Checkpoint: True
model.regression_attention_model.weight

Trained vs Checkpoint: False
Best vs Checkpoint: True
model.regression_attention_model.bias

Trained vs Checkpoint: False
Best vs Checkpoint: True

Model 4
model.bias

Trained vs Checkpoint: False
Best vs Checkpoint: True
model.regression_attention_model.weight

T